# 문맥 보존형 뉴스-KOSIS Top50 in_ready 실험

규칙 기반 정제 Top50 기사에 공통 기사 문맥, 8문장 chunk, 3문장 중첩을 적용하고 `in_ready`까지 실행합니다. HCX 단계는 중단 후 같은 셀을 다시 실행하면 이어받습니다.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

import csv
import json
import os
import random
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

EXPECTED_ARTICLES = 50
REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
RUN_DIR = DRIVE_ROOT / 'runs' / 'contextual_top50_context_v2_8x3'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
ARTICLE_CSV = INPUT_DIR / '뉴스_데이터_규칙기반정제Top50.csv'
EVAL_ARTICLES = ARTICLE_CSV
EARLY_META_CANDIDATES = [
    DRIVE_ROOT / 'runs' / 'early_bge_rag_5000' / 'early_bge_meta_index.csv',
    DRIVE_ROOT / 'runs' / 'early_bge_rag' / 'early_bge_meta_index.csv',
]
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('run:', RUN_DIR)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests>=2.31,<3', 'python-dotenv>=1.0,<2', 'kss>=6,<7',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6', 'transformers>=4.45,<6'
], check=True)
os.chdir(REPO_DIR)
print('commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 원문 기사 CSV

`MyDrive/NLP_05-Team-Project-3/inputs/뉴스_데이터_규칙기반정제Top50.csv`가 없을 때만 업로드 창이 열립니다.

In [ ]:
if not ARTICLE_CSV.exists():
    print('기사 원문 CSV를 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('기사 원문 CSV 한 개만 업로드하세요.')
    shutil.copy2(next(iter(uploaded)), ARTICLE_CSV)
print('article CSV:', ARTICLE_CSV, ARTICLE_CSV.stat().st_size)

In [ ]:
from preprocess_news import read_articles, resolve_columns

articles, fieldnames, encoding = read_articles(ARTICLE_CSV, 'auto')
columns = resolve_columns(fieldnames, {})
if len(articles) != EXPECTED_ARTICLES:
    raise RuntimeError(f'Top50 입력이 아닙니다: {len(articles)} rows')
print(f'articles={len(articles)} encoding={encoding}')
print('resolved columns:', columns)
print('evaluation input:', EVAL_ARTICLES)

In [ ]:
if not os.environ.get('CLOVA_API_KEY'):
    os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY') or ''
if not os.environ.get('CLOVA_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 CLOVA_API_KEY를 등록하세요.')

for path, label in [
    (INDEX_DIR / 'manifest.json', 'BGE manifest'),
    (INDEX_DIR / 'embeddings.npy', 'BGE embeddings'),
    (REPO_DIR / 'data/reference/kosis_table_summary.csv', 'KOSIS table index'),
]:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')
EARLY_META = next((path for path in EARLY_META_CANDIDATES if path.exists()), None)
print('inputs and secrets: ready')
print('early meta:', EARLY_META or '없음')

In [ ]:
command = [
    sys.executable, '-u', str(REPO_DIR / 'run_contextual_news_kosis_pipeline.py'),
    '--articles', str(EVAL_ARTICLES),
    '--table-index', str(REPO_DIR / 'data/reference/kosis_table_summary.csv'),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--device', 'cuda',
    '--chunk-size', '8',
    '--overlap', '3',
    '--lead-sentences', '3',
    '--local-window', '3',
    '--related-limit', '3',
    '--stop-after', 'gate',
]
if EARLY_META:
    command.extend(['--early-meta-index', str(EARLY_META)])
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
def read_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

paths = {
    'sentences': RUN_DIR / '01_sentences.csv',
    'chunks': RUN_DIR / '02_chunks.csv',
    'claim_spans': RUN_DIR / '03_claim_spans.csv',
    'claim_contexts': RUN_DIR / '03_claim_contexts.csv',
    'measurements': RUN_DIR / '05_hcx_measurements.csv',
    'in_ready_all': RUN_DIR / '06_in_ready_all.csv',
}
for label, path in paths.items():
    rows = read_rows(path)
    print(f'{label:18s}: {len(rows):,} rows')

all_rows = read_rows(paths['in_ready_all'])
print('\nin_ready:', Counter(row['in_ready'] for row in all_rows))
print('mapping_gate:', Counter(row['mapping_gate'] for row in all_rows))
print('exclusion:', Counter(row['mapping_exclusion_code'] or 'ELIGIBLE' for row in all_rows).most_common())

In [ ]:
print('\n팀 비교용 최종 파일')
print(RUN_DIR / '06_in_ready_all.csv')
print('\n분리 파일')
for name in ('06_mapping_ready.csv', '06_mapping_enrich.csv', '06_mapping_reject.csv'):
    print(RUN_DIR / name)